# CMP 170HX experiment ledger

**STATUS — VISION VLLM INTEGRATION**  
Last completed gate: real-image functional PASS on the TP4 reference stack after SM80 block-scale, split-sinkhorn, Hadamard, and sparse-attention fallbacks. Text returned exact `OK`; the synthetic gradient image produced the correct color families.  
Current command: pin and statically validate the complete Vision vLLM implementation plus the CMP PP/DSpark semantics.  
Blocker: the active reference runtime has not reached a safe transition boundary; it remains untouched.  
Next gate: exact-source build and config/import validation, then a serialized service transition.

## Attempt table

| Attempt | Phase | Single change | Expected signal | Stop condition | Result | Evidence |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | Import | Source-built SM80 fork | CUDA/custom ops/model registry pass | First decisive import error | PASS | `results/receipts/import-gate.json` |
| 2 | Load | Add F8_E8M0 safetensors mapping | 48 shards + KV allocation + ready | Ready or first decisive failure | Superseded — shard stream passed, engine readiness reached at attempt 12 | `results/receipts/load-gate.json` |
| 12 | Text baseline load | Bind-mount patched DSpark draft loader | Ready + /health 200 after 48/48 shards + draft load | First decisive failure or readiness | PASS | `results/receipts/attempt-12-startup.json` |
| 13 | Vision unit gate | Add SM80 reference fallbacks | Seven focused operator tests pass | First numerical or import mismatch | PASS | `scripts/sm80_unit_test.py` |
| 14 | Vision functional | TP4 reference load + text smoke | Four shards resident; exact `OK` | Load/generation error or accelerator fault | PASS | `results/receipts/vision-reference-smoke.json` |
| 15 | Real image | Data-independent synthetic gradient fixture | Semantically correct color families | Invalid image response or visual mismatch | PASS | `results/receipts/vision-reference-smoke.json` |
| 16 | Reference serving | OpenAI-compatible TP4 broadcast server | models + text + image HTTP 200 | Endpoint, semantic, ownership, or health failure | OWNER ACTIVE — no interruption | `results/receipts/openai-private-live.json` |
| 17 | Vision vLLM integration | Pin exact Vision head and reconcile compiled ABI, PP relay, sparse attention, and three-layer MTP | Static plan and source checks pass | Pin, ABI, path, config, or shape mismatch | REPO VALIDATION IN PROGRESS | `results/receipts/vllm-vision-integration-plan.json` |

Attempt 11 (draft-loader key error) and intermediate attempts are receipted in
`results/receipts/`; measurements for the attempt 12 server are summarized in
`results/receipts/measurements.json`.

In [ ]:
from __future__ import annotations
import json
import os
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RESULTS = ROOT / 'results'
PHASE_ORDER = ['identity', 'storage_preflight', 'load_gate', 'functional_gates', 'measurements', 'publication']
GPU_EXECUTION_ENABLED = os.environ.get('PIXELML_ENABLE_GPU_CELLS') == '1'

def read_json(relative: str) -> dict:
    return json.loads((ROOT / relative).read_text(encoding='utf-8'))

def require_phase(phase_id: str) -> dict:
    state = read_json('results/phase-status.json')
    statuses = {item['id']: item['status'] for item in state['phases']}
    target = PHASE_ORDER.index(phase_id)
    allowed = {'PASS', 'COMPLETE'}
    for prior in PHASE_ORDER[:target]:
        assert statuses[prior] in allowed, f'{prior} blocks {phase_id}: {statuses[prior]}'
    return state

def require_gpu_opt_in() -> None:
    assert GPU_EXECUTION_ENABLED, 'Set PIXELML_ENABLE_GPU_CELLS=1 only after ownership and preflight GO.'


## Vision vLLM integration checkpoint

The exact Vision vLLM head changes both Python and compiled MoE code. The prior CMP patchset is therefore a semantic control, not a bind-mount overlay. PP4 + DSpark k=3 is the first candidate; k=6 is fallback-only, and k=5 is blocked unless the runtime validator proves it valid. No live transition occurs while the reference runtime is active.


In [ ]:
integration = read_json('results/receipts/vllm-vision-integration-plan.json')
assert integration['vision_vllm']['head'] == '2c8af2197ce4b79ce3285724b9a9c69d3f878116'
assert integration['model']['n_mtp_layers'] == 3
candidates = {item['name']: item for item in integration['launch_candidates']}
assert candidates['pp4_k3']['priority'] == 1
assert candidates['pp4_k6']['status'] == 'FALLBACK_ONLY'
assert candidates['pp4_k5']['status'] == 'FORBIDDEN_UNLESS_VALIDATOR_PASS'
integration['delta_table']


## 1. Identity


In [ ]:
manifest = read_json('results/run-manifest.json')
assert manifest['model']['revision'] == '86f746b36186f0e567729a5c06a8c918caba82a9'
manifest


## 2. Storage + preflight


In [ ]:
preflight = read_json('results/receipts/preflight.json')
assert preflight['status'] == 'PASS'
assert preflight['go_no_go'] == 'GO_BOUNDED_COMPATIBILITY_GATE'
preflight


## 3. Load gate


In [ ]:
require_phase('load_gate')
import_gate = read_json('results/receipts/import-gate.json')
load_gate = read_json('results/receipts/load-gate.json')
assert import_gate['status'] == 'PASS'
load_gate


## 4. Functional gates
The text-only vLLM gate and the separate TP4 reference text/image gates are recorded independently.


In [ ]:
require_phase('functional_gates')
vision_smoke = read_json('results/receipts/vision-reference-smoke.json')
assert vision_smoke['text_gate']['status'] == 'PASS'
assert vision_smoke['image_gate']['semantic_check'] == 'PASS'
vision_smoke


## 5. Measurements
The vLLM/DSpark text path has uncached prefill, TTFT, and C1/C2/C4/C8 decode receipts. The vision-capable reference path is functional evidence only; it has no tok/s claim.


In [ ]:
try:
    require_phase('measurements')
    measurement_state = 'READY'
except AssertionError as exc:
    measurement_state = f'BLOCKED: {exc}'
measurement_state


## 6. Publication
Real-image functional evidence and the text-path measurements are complete.
Publication remains pending until the private OpenAI-compatible server passes
models, text, and real-image requests and the public-boundary scan is clean.

In [ ]:
try:
    require_phase('publication')
    publication_state = 'READY'
except AssertionError as exc:
    publication_state = f'BLOCKED: {exc}'
publication_state
